<a href="https://colab.research.google.com/github/Noman654/dataengineer_prep/blob/main/_preview/window_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪟 Window Functions — The Consecutive Months Problem

---

### 💬 New message from Jen (Marketing)

> **Jen** — Monday 9:14 AM
>
> hey!! 👋 quick favor. marketing wants to launch a "thank you for sticking with us" voucher this week.
>
> I need a list of our **loyal regulars** — customers who bought something in **3+ consecutive months**. finance thinks these are our lowest-churn-risk folks and we want to lock them in before summer.
>
> can you pull the list by EOD today? 🙏 you're the best
>
> (ps. walk-ins don't count obviously, only loyalty members with customer_id)

---

Okay. You just opened Slack to this. Let's solve it.

## 🎯 What you'll learn

- How to think in **window functions** instead of self-joins
- The `lag()` pattern for detecting sequences over time
- The classic **gaps and islands** problem — and why it shows up in interviews constantly
- How to answer "N consecutive anything" questions in a handful of lines

**Prerequisites:** You've seen basic `groupBy` and `agg` before.

**Difficulty:** 🟡 Intermediate

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("zephyr_window_functions").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

## 📦 The data

For this notebook we'll use a tiny slice of Zephyr's `transactions` table — 6 customers, 17 transactions, 5 months. Small enough that you can reason about the right answer in your head before we touch any code.

*(In the real repo, this loads from `assets/sample_data/transactions.parquet`. Here it's inline so the notebook is self-contained.)*

In [ ]:
sample_transactions = [
    # Alice (101) — bought every month Jan→May (our clearest regular)
    ("tx_001", 101, "2024-01-15", 12.50),
    ("tx_002", 101, "2024-02-03", 8.75),
    ("tx_003", 101, "2024-03-22", 15.00),
    ("tx_004", 101, "2024-04-11", 9.25),
    ("tx_005", 101, "2024-05-08", 11.00),

    # Bob (102) — Jan, Feb, skipped Mar, Apr — NOT 3 consecutive
    ("tx_006", 102, "2024-01-20", 7.50),
    ("tx_007", 102, "2024-02-14", 8.00),
    ("tx_008", 102, "2024-04-02", 10.25),

    # Carol (103) — Feb, Mar, Apr — exactly 3 consecutive ✓
    ("tx_009", 103, "2024-02-05", 6.50),
    ("tx_010", 103, "2024-03-18", 12.00),
    ("tx_011", 103, "2024-04-27", 9.75),

    # Dave (104) — one transaction, one month
    ("tx_012", 104, "2024-03-10", 5.50),

    # Eve (105) — two transactions, both in March — same month twice
    ("tx_013", 105, "2024-03-05", 8.00),
    ("tx_014", 105, "2024-03-25", 11.50),

    # Frank (106) — Jan, Feb, Mar then nothing — exactly 3 ✓
    ("tx_015", 106, "2024-01-08", 7.00),
    ("tx_016", 106, "2024-02-19", 9.50),
    ("tx_017", 106, "2024-03-30", 13.25),
]

transactions = (
    spark.createDataFrame(
        sample_transactions,
        ["tx_id", "customer_id", "ts", "total_amount"],
    )
    .withColumn("ts", F.to_timestamp("ts"))
)

transactions.show()

### 🧠 Before writing any code — figure out the answer by hand

Seriously, pause for 30 seconds. Which customers should end up on Jen's list?

<details>
<summary>Click for the expected answer</summary>

- **Alice (101)** — bought every month Jan → May ✓
- **Carol (103)** — bought Feb, Mar, Apr ✓
- **Frank (106)** — bought Jan, Feb, Mar ✓

Bob skipped March. Dave bought once. Eve bought twice but in the same month.

Keep this in your head — when your query runs, if it returns anything else, you know it's wrong.
</details>

## 🤔 Pause. How would you even do this?

Before scrolling down, actually think about it. How do you find "bought something in **3 consecutive months**"?

<details>
<summary>Hint 1</summary>

You can't do this with `groupBy` alone. `groupBy(customer_id).count()` tells you *how many* months they bought in, not whether those months were *consecutive*.
</details>

<details>
<summary>Hint 2</summary>

You need each row to "know" what came before it. That's exactly what **window functions** do — they let a row peek at its neighbors inside a sorted group.
</details>

<details>
<summary>Hint 3</summary>

Key function: `F.lag(col, n)` — returns the value of `col` from `n` rows earlier *within a window*. If you sort transactions by month per customer, `lag(month, 1)` gives you the previous month they bought in.
</details>

## 💡 Walkthrough

### Step 1 — Reduce to one row per customer per month

We don't care how many times Alice bought in January, only *that* she did. Collapse down:

In [ ]:
monthly = (
    transactions
    .withColumn("month", F.date_trunc("month", "ts"))
    .select("customer_id", "month")
    .distinct()
    .orderBy("customer_id", "month")
)

monthly.show()

Eve's two March transactions collapsed to one row. Good — that's what `.distinct()` did.

### Step 2 — For each row, what was the previous month?

Here's where windows come in. We define a window that says *"group by customer_id, and within each group, sort by month"*:

```python
w = Window.partitionBy("customer_id").orderBy("month")
```

Now any window function (`lag`, `lead`, `row_number`, `sum`, `rank`...) applied with `.over(w)` operates **inside that sorted group**. `lag(month, 1).over(w)` = *"for each row, what was `month` one row ago, within this customer's sorted timeline?"*

In [ ]:
w = Window.partitionBy("customer_id").orderBy("month")

with_prev = monthly.withColumn("prev_month", F.lag("month", 1).over(w))

with_prev.show()

Look at Alice's rows. Row by row, `prev_month` now tells her *exactly* what month she bought in just before this one. The first row of each customer has `prev_month = null` because there's nothing before it — that's normal.

### Step 3 — Is each row consecutive with the previous?

If `month - prev_month` is exactly 1 month → consecutive. Otherwise → gap.

In [ ]:
with_flag = with_prev.withColumn(
    "is_consecutive",
    F.when(F.months_between("month", "prev_month") == 1, 1).otherwise(0),
)

with_flag.show()

### Step 4 — The trick: spotting a streak of 3

Here's the clever bit. `is_consecutive = 1` means "this month touches the previous one". But that's only a **streak of 2**. To find a streak of **3 consecutive months**, we need a row where `is_consecutive = 1` **and** the row just before *also* had `is_consecutive = 1`.

In other words: lag the flag itself.

In [ ]:
final = (
    with_flag
    .withColumn("prev_consecutive", F.lag("is_consecutive", 1).over(w))
    .filter((F.col("is_consecutive") == 1) & (F.col("prev_consecutive") == 1))
    .select("customer_id")
    .distinct()
)

final.show()

## 📤 Ship it to Jen

```
customer_id
-----------
101   (Alice)
103   (Carol)
106   (Frank)
```

Exactly what we predicted by hand. Copy, paste into Slack, mission accomplished.

> **You** → Jen: here's the list, 3 customers fit. LMK if you want it broader (2 consecutive months) or different time window 👍
>
> **Jen**: omg lifesaver ty 🙌

---

### 🧠 The pattern you just learned

This "lag-and-flag" approach solves a **huge** family of problems:

- 3 consecutive months of purchases ✓
- Login streaks (daily active users N days in a row)
- Price drops on N consecutive days
- "First time a user did X after doing Y"
- Session detection (gap of >30 min = new session)

All the same pattern: **sort by time, lag the relevant column, compare neighbors.** Once you see it, you see it everywhere.

## 🎯 Your turn — Jen came back

> **Jen** — 2:30 PM
>
> ok different angle. finance is worried about **churn**. can you find customers whose **monthly spending dropped for 3 months in a row**? like Jan = $50, Feb = $30, Mar = $15. those are the ones we want to save before they leave us entirely.

Same dataset. Same window trick. Different comparison. (Hint: you're now comparing *amounts* across months, not *existence* of months.)

Try it in the cell below. No solution at the bottom of the notebook — that's deliberate. Struggle with it for 10 minutes before you look anything up.

In [ ]:
# your code here




## 🏆 Boss Level — Marcus wants top-3 products per store per quarter

> **Marcus (CFO)** — Friday 6:04 PM (of course)
>
> Need the top 3 products by revenue for each store, for each quarter of 2024. Executive deck Monday. Use whatever tricks you want.

This one needs `row_number()` or `rank()` or `dense_rank()` — and the difference between them **actually matters here**.

- `row_number()` — unique sequential numbers, even for ties → (1, 2, 3, 4)
- `rank()` — ties share a rank, then skip → (1, 2, 2, 4)
- `dense_rank()` — ties share a rank, no skip → (1, 2, 2, 3)

**Think before you code:** if two products tie for #3 revenue, which function gives Marcus the answer he probably wants? Would he rather see 2 products or 3 in the top-3? There's no single right answer — the point is to *notice the ambiguity and ask*.

**Stretch question:** the real `transaction_items` table has ~1.5M rows. Does your query scale? How would you partition the data in production to keep the window efficient?

## 📚 Further reading (later, not now)

- [Spark docs — Window functions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)
- *Gaps and Islands* by Itzik Ben-Gan — the canonical essay on this pattern
- Databricks blog on window function performance — matters once you're past ~10M rows

That's enough window functions for one day. Save your notebook. Go get a coffee (Zephyr brand, obviously).